In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

import datasets
import json
import numpy as np
import pandas as pd
from collections import defaultdict

/home/artem.shelmanov/miniconda3/envs/uhead/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch

device = 'cuda'

model = LLaDAModelLM.from_pretrained('GSAI-ML/LLaDA-1.5', trust_remote_code=True, torch_dtype=torch.bfloat16).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained('GSAI-ML/LLaDA-1.5', trust_remote_code=True)
prompt = "Given the following question, reason and give a final answer to the question. Your response should end with \"The answer is [answer]\" where [answer] is the response to the question.\n\nHere are a few examples of questions and answers:\n\nQuestion: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\nAnswer: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.\n\nQuestion: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\nAnswer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.\n\nQuestion: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\nAnswer: Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. The answer is 39.\n\nQuestion: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?\nAnswer: Jason started with 20 lollipops. Then he had 12 after giving some to Denny. So he gave Denny 20 - 12 = 8. The answer is 8.\n\nQuestion: Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?\nAnswer: Shawn started with 5 toys. If he got 2 toys each from his mom and dad, then that is 4 more toys. 5 + 4 = 9. The answer is 9.\n\nQuestion: There were nine computers in the server room. Five more computers were installed each day, from monday to thursday. How many computers are now in the server room?\nAnswer: There were originally 9 computers. For each of 4 days, 5 more computers were added. So 5 * 4 = 20 computers were added. 9 + 20 is 29. The answer is 29.\n\nQuestion: Michael had 58 golf balls. On tuesday, he lost 23 golf balls. On wednesday, he lost 2 more. How many golf balls did he have at the end of wednesday?\nAnswer: Michael started with 58 golf balls. After losing 23 on tuesday, he had 58 - 23 = 35. After losing 2 more, he had 35 - 2 = 33 golf balls. The answer is 33.\n\nQuestion: Olivia has $23. She bought five bagels for $3 each. How much money does she have left?\nAnswer: Olivia had 23 dollars. 5 bagels for 3 dollars each will be 5 x 3 = 15 dollars. So she has 23 - 15 dollars left. 23 - 15 is 8. The answer is 8.\n\nNow answer the following question.\n\nQuestion: John collects peaches for 3 hours.  He can collect 2 peaches a minute.  How many peaches does he collect?\nAnswer:"

# Add special tokens for the Instruct model. The Base model does not require the following two lines.
m = [{"role": "user", "content": prompt}, ]
prompt = tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=False)

input_ids = tokenizer(prompt)['input_ids']
input_ids = torch.tensor(input_ids).to(device).unsqueeze(0)

Loading checkpoint shards: 100%|██████████| 6/6 [00:01<00:00,  4.61it/s]


In [5]:
import torch

cov = torch.load("cov_scaled.pt")

In [10]:
eigenvalues = torch.linalg.eigvalsh(cov.double().cpu())

_LinAlgError: linalg.eigh: The algorithm failed to converge because the input matrix is ill-conditioned or has too many repeated eigenvalues (error code: 4094).

In [12]:
cov.float().

tensor([[ 9.0698e-02, -9.0179e-03, -5.7526e-03,  ..., -8.5831e-03,
         -8.3447e-05,  2.0351e-03],
        [-9.0179e-03,  9.1370e-02, -8.3313e-03,  ..., -1.2245e-03,
          8.2779e-03, -1.6718e-03],
        [-5.7526e-03, -8.3313e-03,  9.8511e-02,  ..., -6.0616e-03,
          7.1907e-04, -1.0204e-03],
        ...,
        [-8.5831e-03, -1.2245e-03, -6.0616e-03,  ...,  1.0919e-01,
         -9.1248e-03, -1.3084e-02],
        [-8.3447e-05,  8.2779e-03,  7.1907e-04,  ..., -9.1248e-03,
          1.0168e-01,  1.8806e-03],
        [ 2.0351e-03, -1.6718e-03, -1.0204e-03,  ..., -1.3084e-02,
          1.8806e-03,  1.2659e-01]], device='cuda:0')

In [13]:
torch.isnan(cov).sum().item()

8190

In [16]:
torch.isinf(cov).sum().item()

1

In [20]:
torch.linalg.eigh(torch.nan_to_num(cov, nan=0.0, posinf=0.0, neginf=0.0).float())

torch.return_types.linalg_eigh(
eigenvalues=tensor([-1.1067e-03, -8.5164e-04, -6.9493e-04,  ...,  9.0021e+00,
         1.1842e+01,  2.0824e+01], device='cuda:0'),
eigenvectors=tensor([[ 0.0004,  0.0120, -0.0268,  ...,  0.0095, -0.0026,  0.0133],
        [ 0.0275, -0.0062,  0.0183,  ..., -0.0094, -0.0046, -0.0173],
        [-0.0093, -0.0102,  0.0072,  ..., -0.0095,  0.0023,  0.0011],
        ...,
        [-0.0025, -0.0157,  0.0083,  ...,  0.0007, -0.0141,  0.0043],
        [-0.0097,  0.0034,  0.0203,  ..., -0.0137,  0.0007, -0.0136],
        [ 0.0086, -0.0376,  0.0424,  ...,  0.0014, -0.0084, -0.0173]],
       device='cuda:0'))

In [4]:
model.config

LLaDAConfig {
  "_attn_implementation_autoset": true,
  "activation_type": "silu",
  "alibi": false,
  "alibi_bias_max": 8.0,
  "architectures": [
    "LLaDAModelLM"
  ],
  "attention_dropout": 0.0,
  "attention_layer_norm": false,
  "attention_layer_norm_with_affine": true,
  "auto_map": {
    "AutoConfig": "GSAI-ML/LLaDA-1.5--configuration_llada.LLaDAConfig",
    "AutoModel": "GSAI-ML/LLaDA-1.5--modeling_llada.LLaDAModelLM",
    "AutoModelForCausalLM": "GSAI-ML/LLaDA-1.5--modeling_llada.LLaDAModelLM"
  },
  "bias_for_layer_norm": false,
  "block_group_size": 1,
  "block_type": "llama",
  "d_model": 4096,
  "embedding_dropout": 0.0,
  "embedding_size": 126464,
  "eos_token_id": 126081,
  "flash_attention": false,
  "include_bias": false,
  "include_qkv_bias": false,
  "init_cutoff_factor": null,
  "init_device": "meta",
  "init_fn": "mitchell",
  "init_std": 0.02,
  "input_emb_norm": false,
  "layer_norm_type": "rms",
  "layer_norm_with_affine": true,
  "mask_token_id": 126336,
  "max

In [11]:
from llada.generate import *

In [12]:
steps = 1024

In [37]:
%%timeit

out = generate(model, input_ids, steps=steps, gen_length=steps, block_length=32, temperature=0., remasking='low_confidence')

44.1 s ± 274 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [38]:
%%timeit

out = generate(model, input_ids, steps=steps, gen_length=steps, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9)

3.94 s ± 3.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [41]:
%%timeit

out = generate_with_dual_cache(model, input_ids, steps=steps, gen_length=steps, block_length=32, temperature=0., remasking='low_confidence')

32.3 s ± 390 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [39]:
%%timeit

out = generate_with_dual_cache(model, input_ids, steps=steps, gen_length=steps, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

3.25 s ± 375 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [7]:
%%timeit

out = generate_with_dual_cache(model, input_ids, steps=steps, gen_length=steps, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9)

3.32 s ± 164 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [40]:
%%timeit

out = generate_with_prefix_cache(model, input_ids, steps=steps, gen_length=steps, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

3.15 s ± 6.98 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [42]:
%%timeit

with torch.inference_mode():
    nvtx.range_push("INFER")

    out = generate_with_dual_cache(model, input_ids, steps=steps, gen_length=steps, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

    torch.cuda.synchronize()
    nvtx.range_pop()

3.13 s ± 430 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [26]:
steps, gen_length, block_length = 128, 512, 32

In [9]:
%%timeit

out = generate(model, input_ids, steps=steps, gen_length=gen_length, block_length=32, temperature=0., remasking='low_confidence')

6.98 s ± 163 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [10]:
%%timeit

out = generate(model, input_ids, steps=steps, gen_length=gen_length, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9)

5.89 s ± 251 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [11]:
%%timeit

out = generate_with_dual_cache(model, input_ids, steps=gen_length, gen_length=steps, block_length=32, temperature=0., remasking='low_confidence')

4.15 s ± 255 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
%%timeit

out = generate_with_dual_cache(model, input_ids, steps=steps, gen_length=gen_length, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9)

2.64 s ± 165 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [24]:
%%timeit

out = generate_with_dual_cache(model, input_ids, steps=steps, gen_length=gen_length, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

1.5 s ± 309 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [25]:
%%timeit

out = generate_with_prefix_cache(model, input_ids, steps=steps, gen_length=gen_length, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

1.44 s ± 3.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [27]:
%%timeit

with torch.inference_mode():
    nvtx.range_push("INFER")

    out = generate_with_prefix_cache(model, input_ids, steps=steps, gen_length=steps, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

    torch.cuda.synchronize()
    nvtx.range_pop()

804 ms ± 32.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [28]:
%%timeit

with torch.inference_mode():
    nvtx.range_push("INFER")

    out = generate_with_dual_cache(model, input_ids, steps=steps, gen_length=steps, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

    torch.cuda.synchronize()
    nvtx.range_pop()

848 ms ± 18.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [29]:
out = generate_with_dual_cache(model, input_ids, steps=steps, gen_length=gen_length, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9)

In [30]:
tokenizer.batch_decode(out[0], skip_special_tokens=True)

['user\n\nGiven the following question, reason and give a final answer to the question. Your response should end with "The answer is [answer]" where [answer] is the response to the question.\n\nHere are a few examples of questions and answers:\n\nQuestion: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\nAnswer: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.\n\nQuestion: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\nAnswer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.\n\nQuestion: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\nAnswer: Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 7

In [31]:
tokenizer.batch_decode(out[0])

['<|startoftext|><|start_header_id|>user<|end_header_id|>\n\nGiven the following question, reason and give a final answer to the question. Your response should end with "The answer is [answer]" where [answer] is the response to the question.\n\nHere are a few examples of questions and answers:\n\nQuestion: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\nAnswer: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.\n\nQuestion: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\nAnswer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.\n\nQuestion: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\nAnswer: Originally, Leah had 32 chocolates. 

In [4]:
path1 = "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/['gsm8k', 'main']/2025-06-26/15-22-42/ue_manager_seed1"
path2 = "./workdir/output/qa/GSAI-ML/LLaDA-8B-Instruct/['gsm8k', 'main']/2025-12-11/21-13-27/ue_manager_seed1"

In [5]:
data1 = torch.load(path1, weights_only=False)
data2 = torch.load(path2, weights_only=False)

In [6]:
data1["stats"]["greedy_texts"][0]

" Janet's ducks lay 16 eggs per day. She eats 3 eggs for breakfast and bakes 4 eggs for her friends. So she uses 3 + 4 = 7 eggs per day. She has 16 - 7 = 9 eggs left to sell. She sells these 9 eggs for $2 each, so she makes 9 * $2 = $18 per day. The answer is $18."

In [7]:
data2["stats"]["greedy_texts"][0]

" Janet's ducks lay 16 eggs per day. She eats 3 eggs for breakfast and bakes 4 eggs for her friends<|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|> 3 + 4 = 7 eggs. The remaining eggs are 16 - 7 = 9<|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|><|mdm_mask|>"

In [8]:
data1["stats"]["input_texts"][0] == data2["stats"]["input_texts"][0]

True

In [9]:
text = "Given the following question, reason and give a final answer to the question. Your response should end with \"The answer is [answer]\" where [answer] is the response to the question.\n\nHere are a few examples of questions and answers:\n\nQuestion: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\nAnswer: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.\n\nQuestion: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\nAnswer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.\n\nQuestion: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\nAnswer: Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. The answer is 39.\n\nQuestion: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?\nAnswer: Jason started with 20 lollipops. Then he had 12 after giving some to Denny. So he gave Denny 20 - 12 = 8. The answer is 8.\n\nQuestion: Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?\nAnswer: Shawn started with 5 toys. If he got 2 toys each from his mom and dad, then that is 4 more toys. 5 + 4 = 9. The answer is 9.\n\nQuestion: There were nine computers in the server room. Five more computers were installed each day, from monday to thursday. How many computers are now in the server room?\nAnswer: There were originally 9 computers. For each of 4 days, 5 more computers were added. So 5 * 4 = 20 computers were added. 9 + 20 is 29. The answer is 29.\n\nQuestion: Michael had 58 golf balls. On tuesday, he lost 23 golf balls. On wednesday, he lost 2 more. How many golf balls did he have at the end of wednesday?\nAnswer: Michael started with 58 golf balls. After losing 23 on tuesday, he had 58 - 23 = 35. After losing 2 more, he had 35 - 2 = 33 golf balls. The answer is 33.\n\nQuestion: Olivia has $23. She bought five bagels for $3 each. How much money does she have left?\nAnswer: Olivia had 23 dollars. 5 bagels for 3 dollars each will be 5 x 3 = 15 dollars. So she has 23 - 15 dollars left. 23 - 15 is 8. The answer is 8.\n\nNow answer the following question.\n\nQuestion: A basket of green food costs $25 and a basket of red food costs $18. If you buy 3 baskets of green food and red food, how much will you have to pay in total if you get $2 off for each basket of red food?\nAnswer:"
m = [{"role": "user", "content": text}, ]
prompt = tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=False)

input_ids = tokenizer(prompt)['input_ids']
input_ids = torch.tensor(input_ids).to(device).unsqueeze(0)

In [11]:
out = generate_with_dual_cache(model, input_ids, steps=128, gen_length=256, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)
# out = generate_with_prefix_cache(model, input_ids, steps=steps, gen_length=gen_length, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)
print(tokenizer.batch_decode(out[0])[0].split("<|endoftext|>")[0])

<|startoftext|><|start_header_id|>user<|end_header_id|>

Given the following question, reason and give a final answer to the question. Your response should end with "The answer is [answer]" where [answer] is the response to the question.

Here are a few examples of questions and answers:

Question: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
Answer: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.

Question: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
Answer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.

Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
Answer: Originally, Leah had 32 chocolates. Her sister had 

In [35]:
with torch.inference_mode():
    nvtx.range_push("INFER")

    out = generate_with_dual_cache(model, input_ids, steps=128, gen_length=256, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

    torch.cuda.synchronize()
    nvtx.range_pop()
print(tokenizer.batch_decode(out[0])[0].split("<|endoftext|>")[0])

<|startoftext|><|start_header_id|>user<|end_header_id|>

Given the following question, reason and give a final answer to the question. Your response should end with "The answer is [answer]" where [answer] is the response to the question.

Here are a few examples of questions and answers:

Question: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
Answer: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.

Question: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
Answer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.

Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
Answer: Originally, Leah had 32 chocolates. Her sister had 

In [94]:
print(input_ids)

tensor([[126080, 126346,   3840, 126347,    198,    198,  19781,    268,   2538,
           3244,     11,   3401,    301,   2697,    259,   2753,   3766,    297,
            268,   3244,     13,   3690,   4031,   1329,   1389,    391,    477,
            678,   3766,    341,    831,  31113,  39941,   1387,    831,  31113,
             60,    341,    268,   4031,    297,    268,   3244,     13,    198,
            198,   6952,    449,    259,   2351,   9110,    300,   4294,    301,
          10891,     25,    198,    198,  17188,     25,   2263,    449,    220,
             16,     20,  10864,    296,    268, 124721,     13,  38578,   8673,
            627,   5975,  10864,    296,    268, 124721,   3342,     13,   4474,
            792,    449,   3226,     11,    947,    627,    367,    220,     17,
             16,  10864,     13,   2071,   1494,  10864,   1556,    268, 124721,
           8673,   5975,   3342,     30,    198,  19002,     25,   2263,    449,
            220,     16,    

In [65]:
%%timeit

with torch.inference_mode():
    nvtx.range_push("INFER")

    out = generate_with_dual_cache(model, input_ids, steps=steps, gen_length=steps, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

    torch.cuda.synchronize()
    nvtx.range_pop()

1.47 s ± 289 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [66]:
%%timeit

with torch.inference_mode():
    nvtx.range_push("INFER")

    out = generate_with_prefix_cache(model, input_ids, steps=steps, gen_length=steps, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

    torch.cuda.synchronize()
    nvtx.range_pop()

1.04 s ± 13.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [67]:
%%timeit

out = generate_with_prefix_cache(model, input_ids, steps=steps, gen_length=gen_length, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

1.65 s ± 3.61 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [68]:
out = generate_with_prefix_cache(model, input_ids, steps=steps, gen_length=gen_length, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)
print(tokenizer.batch_decode(out[0])[0].split("<|endoftext|>")[0])

<|startoftext|><|start_header_id|>user<|end_header_id|>

Given the following question, reason and give a final answer to the question. Your response should end with "The answer is [answer]" where [answer] is the response to the question.

Here are a few examples of questions and answers:

Question: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
Answer: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.

Question: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
Answer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.

Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
Answer: Originally, Leah had 32 chocolates. Her sister had 

In [69]:
with torch.inference_mode():
    nvtx.range_push("INFER")

    out = generate_with_prefix_cache(model, input_ids, steps=steps, gen_length=steps, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

    torch.cuda.synchronize()
    nvtx.range_pop()
print(tokenizer.batch_decode(out[0])[0].split("<|endoftext|>")[0])

<|startoftext|><|start_header_id|>user<|end_header_id|>

Given the following question, reason and give a final answer to the question. Your response should end with "The answer is [answer]" where [answer] is the response to the question.

Here are a few examples of questions and answers:

Question: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
Answer: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.

Question: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
Answer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.

Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
Answer: Originally, Leah had 32 chocolates. Her sister had 

In [49]:
%%timeit

with torch.inference_mode():
    nvtx.range_push("INFER")

    out = generate_with_dual_cache(model, input_ids, steps=steps, gen_length=steps, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

    torch.cuda.synchronize()
    nvtx.range_pop()

1.42 s ± 15.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
steps, gen_length, block_length = 128, 20, 5 

In [66]:
%%timeit

out = generate(model, input_ids, steps=steps, gen_length=gen_length, block_length=block_length, temperature=0., remasking='low_confidence')

460 ms ± 1.72 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [67]:
%%timeit

out = generate(model, input_ids, steps=steps, gen_length=gen_length, block_length=block_length, temperature=0., remasking='low_confidence', threshold=0.9)

297 ms ± 1.83 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [68]:
%%timeit

out = generate_with_dual_cache(model, input_ids, steps=steps, gen_length=gen_length, block_length=block_length, temperature=0., remasking='low_confidence')

575 ms ± 7.12 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [69]:
%%timeit

out = generate_with_dual_cache(model, input_ids, steps=steps, gen_length=gen_length, block_length=block_length, temperature=0., remasking='low_confidence', threshold=0.9)

333 ms ± 5.57 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [70]:
%%timeit

out = generate_with_dual_cache(model, input_ids, steps=steps, gen_length=gen_length, block_length=block_length, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

253 ms ± 7.51 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [7]:
%%timeit

out = generate_with_prefix_cache(model, input_ids, steps=steps, gen_length=gen_length, block_length=block_length, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

193 ms ± 1.26 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [8]:
%%timeit

out = generate_with_prefix_cache(model, input_ids, steps=steps, gen_length=gen_length, block_length=block_length, temperature=0., remasking='low_confidence', threshold=0.9)

264 ms ± 2.05 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [63]:
%%timeit

with torch.inference_mode():
    nvtx.range_push("INFER")

    out = generate_with_dual_cache(model, input_ids, steps=steps, gen_length=steps, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

    torch.cuda.synchronize()
    nvtx.range_pop()

452 ms ± 12.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [15]:
ids = torch.tensor([[126080, 126346,   3840, 126347,    198,    198,  19781,    268,   2538,
           3244,     11,   3401,    301,   2697,    259,   2753,   3766,    297,
            268,   3244,     13,   3690,   4031,   1329,   1389,    391,    477,
            678,   3766,    341,    831,  31113,  39941,   1387,    831,  31113,
             60,    341,    268,   4031,    297,    268,   3244,     13,    198,
            198,   6952,    449,    259,   2351,   9110,    300,   4294,    301,
          10891,     25,    198,    198,  17188,     25,   2263,    449,    220,
             16,     20,  10864,    296,    268, 124721,     13,  38578,   8673,
            627,   5975,  10864,    296,    268, 124721,   3342,     13,   4474,
            792,    449,   3226,     11,    947,    627,    367,    220,     17,
             16,  10864,     13,   2071,   1494,  10864,   1556,    268, 124721,
           8673,   5975,   3342,     30,    198,  19002,     25,   2263,    449,
            220,     16,     20,  10864,  14554,     13,   5168,    947,    929,
            220,     17,     16,  10864,   1459,    971,    778,    929,  34455,
             13,   2237,    947,   2022,    561,   1007,    220,     17,     16,
            537,    220,     16,     20,    373,    220,     21,     13,    468,
           3766,    341,    220,     21,     13,    198,    198,  17188,     25,
           1538,    947,    449,    220,     18,  10537,    296,    268,  13542,
           2304,    301,    220,     17,    778,  10537,  17261,     11,   1099,
           1494,  10537,    449,    296,    268,  13542,   2304,     30,    198,
          19002,     25,   2263,    449,  14554,    220,     18,  10537,     13,
            220,     17,    778,  10537,  17261,     13,    220,     18,    781,
            220,     17,    373,    220,     20,     13,    468,   3766,    341,
            220,     20,     13,    198,    198,  17188,     25,  71939,   1028,
            220,     18,     17,  89513,    301,   1094,  12672,   1028,    220,
             19,     17,     13,   1538,    792,  29183,    220,     18,     20,
             11,   1099,   1494,   9318,    640,    792,    561,   2914,    296,
           3886,     30,    198,  19002,     25,  53348,     11,  71939,   1028,
            220,     18,     17,  89513,     13,   6617,  12672,   1028,    220,
             19,     17,     13,   2237,    296,   3886,    792,   1028,    220,
             18,     17,    781,    220,     19,     17,    373,    220,     22,
             19,     13,   4474,  11110,    220,     18,     20,     11,    792,
           1028,    220,     22,     19,    537,    220,     18,     20,    373,
            220,     18,     24,     13,    468,   3766,    341,    220,     18,
             24,     13,    198,    198,  17188,     25,  22239,   1028,    220,
             17,     15,    322, 107300,   3984,     13,   1290,   6655, 122419,
            971,    322, 107300,   3984,     13,   4948,  22239,    723,    220,
             16,     17,    322, 107300,   3984,     13,   2071,   1494,    322,
         107300,   3984,   1556,  22239,   2697,    297, 122419,     30,    198,
          19002,     25,  22239,   4057,    391,    220,     17,     15,    322,
         107300,   3984,     13,   5168,    464,   1028,    220,     16,     17,
           1459,   6787,    971,    297, 122419,     13,   2237,    464,   6655,
         122419,    220,     17,     15,    537,    220,     16,     17,    373,
            220,     23,     13,    468,   3766,    341,    220,     23,     13,
            198,    198,  17188,     25,  66264,    723,   4428,  23628,     13,
           1741,   7791,     11,    464,   2686,   1307,  23628,   1671,    546,
            900,   3689,    301,  17246,     13,   2071,   1494,  23628,   1543,
            464,    561,   1550,     30,    198,  19002,     25,  66264,   4057,
            391,    220,     20,  23628,     13,   1538,    464,   2686,    220,
             17,  23628,   1671,    546,    900,   3689,    301,  17246,     11,
           1310,    378,    341,    220,     19,    778,  23628,     13,    220,
             20,    781,    220,     19,    373,    220,     24,     13,    468,
           3766,    341,    220,     24,     13,    198,    198,  17188,     25,
           2263,    929,  13097,  18568,    296,    268,   6054,   3284,     13,
          21967,    778,  18568,    929,  11115,   1671,   1813,     11,    546,
         123884,    297,    308,   8805,     13,   2071,   1494,  18568,    449,
           1550,    296,    268,   6054,   3284,     30,    198,  19002,     25,
           2263,    929,  14554,    220,     24,  18568,     13,   1741,   1671,
            300,    220,     19,   2826,     11,    220,     20,    778,  18568,
            929,   4496,     13,   2237,    220,     20,    500,    220,     19,
            373,    220,     17,     15,  18568,    929,   4496,     13,    220,
             24,    781,    220,     17,     15,    341,    220,     17,     24,
             13,    468,   3766,    341,    220,     17,     24,     13,    198,
            198,  17188,     25,   9529,   1028,    220,     20,     23,  17048,
          22493,     13,   2490,    257,   8816,     11,    464,   6040,    220,
             17,     18,  17048,  22493,     13,   2490,    274,   9024,     11,
            464,   6040,    220,     17,    778,     13,   2071,   1494,  17048,
          22493,   1556,    464,    561,    482,    268,   1389,    300,    274,
           9024,     30,    198,  19002,     25,   9529,   4057,    391,    220,
             20,     23,  17048,  22493,     13,   4474,  14261,    220,     17,
             18,    366,    257,   8816,     11,    464,   1028,    220,     20,
             23,    537,    220,     17,     18,    373,    220,     18,     20,
             13,   4474,  14261,    220,     17,    778,     11,    464,   1028,
            220,     18,     20,    537,    220,     17,    373,    220,     18,
             18,  17048,  22493,     13,    468,   3766,    341,    220,     18,
             18,     13,    198,    198,  17188,     25,  56547,    723,    558,
             17,     18,     13,   2608,  10695,   4428,   9549,   1631,    352,
            558,     18,   1671,     13,   2071,   1670,   3141,   1543,   1285,
            561,   2914,     30,    198,  19002,     25,  56547,   1028,    220,
             17,     18,  12161,     13,    220,     20,   9549,   1631,    352,
            220,     18,  12161,   1671,    627,    367,    220,     20,   1529,
            220,     18,    373,    220,     16,     20,  12161,     13,   2237,
           1285,    723,    220,     17,     18,    537,    220,     16,     20,
          12161,   2914,     13,    220,     17,     18,    537,    220,     16,
             20,    341,    220,     23,     13,    468,   3766,    341,    220,
             23,     13,    198,    198,   6301,   3766,    268,   2538,   3244,
             13,    198,    198,  17188,     25,   4272,   1654,    723,    220,
             18,  20258,     13,   2608,  37696,   1094,  20258,  11553,    259,
           1813,    391,    220,     21,     15,  32692,    300,   7748,   3199,
             13,   2071,   1494,   2826,    627,    220,     22,     17,     15,
          32692,    300,   7748,   3199,   1948,     30,    198,  19002,     25,
         126348, 126346,    598,  10450, 126347,    198,    198]]).cuda()

In [16]:
inp = tokenizer.batch_decode(ids)
print(inp[0])

<|startoftext|><|start_header_id|>user<|end_header_id|>

Given the following question, reason and give a final answer to the question. Your response should end with "The answer is [answer]" where [answer] is the response to the question.

Here are a few examples of questions and answers:

Question: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
Answer: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.

Question: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
Answer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.

Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
Answer: Originally, Leah had 32 chocolates. Her sister had 

In [17]:
inp[0].split("<|end_header_id|>")[1].split("<|eot_id|>")[0]

'\n\nGiven the following question, reason and give a final answer to the question. Your response should end with "The answer is [answer]" where [answer] is the response to the question.\n\nHere are a few examples of questions and answers:\n\nQuestion: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\nAnswer: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.\n\nQuestion: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\nAnswer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.\n\nQuestion: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\nAnswer: Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. Af

In [18]:
with torch.inference_mode():
    nvtx.range_push("INFER")

    out = generate_with_dual_cache(model, ids, steps=128, gen_length=256, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

    torch.cuda.synchronize()
    nvtx.range_pop()
print(tokenizer.batch_decode(out[0])[0].split("<|endoftext|>")[0])

<|startoftext|><|start_header_id|>user<|end_header_id|>

Given the following question, reason and give a final answer to the question. Your response should end with "The answer is [answer]" where [answer] is the response to the question.

Here are a few examples of questions and answers:

Question: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
Answer: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.

Question: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
Answer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.

Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
Answer: Originally, Leah had 32 chocolates. Her sister had 

In [26]:
%%timeit

with torch.inference_mode():
    nvtx.range_push("INFER")

    out = generate_with_dual_cache(model, ids, steps=256, gen_length=256, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

    torch.cuda.synchronize()
    nvtx.range_pop()
print(tokenizer.batch_decode(out[0])[0].split("<|endoftext|>")[0])

<|startoftext|><|start_header_id|>user<|end_header_id|>

Given the following question, reason and give a final answer to the question. Your response should end with "The answer is [answer]" where [answer] is the response to the question.

Here are a few examples of questions and answers:

Question: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
Answer: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.

Question: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
Answer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.

Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
Answer: Originally, Leah had 32 chocolates. Her sister had 

In [19]:
%%timeit

with torch.inference_mode():
    nvtx.range_push("INFER")

    out = generate_with_prefix_cache(model, ids, steps=128, gen_length=256, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

    torch.cuda.synchronize()
    nvtx.range_pop()
print(tokenizer.batch_decode(out[0])[0].split("<|endoftext|>")[0])

<|startoftext|><|start_header_id|>user<|end_header_id|>

Given the following question, reason and give a final answer to the question. Your response should end with "The answer is [answer]" where [answer] is the response to the question.

Here are a few examples of questions and answers:

Question: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
Answer: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.

Question: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
Answer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.

Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
Answer: Originally, Leah had 32 chocolates. Her sister had 

In [21]:
%%timeit

with torch.inference_mode():
    nvtx.range_push("INFER")

    out = generate_with_dual_cache(model, ids, steps=128, gen_length=256, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)

    torch.cuda.synchronize()
    nvtx.range_pop()

1.64 s ± 20.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [10]:
# pip install -U transformers accelerate torch

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.31it/s]


In [11]:
messages = [
    {"role": "user", "content": inp[0].split("<|end_header_id|>")[1].split("<|eot_id|>")[0]},
]

# Convert chat messages to the model's expected prompt format
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

In [14]:
%%timeit

with torch.no_grad():
    outputs = model.generate(
        inputs,
        max_new_tokens=256,
        do_sample=False,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.05,
        eos_token_id=tokenizer.eos_token_id,
    )

/home/jovyan/.mlspace/envs/d-llm/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jovyan/.mlspace/envs/d-llm/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected b

1.68 s ± 10.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
out = generate_with_dual_cache(model, input_ids, steps=128, gen_length=256, block_length=32, temperature=0., remasking='low_confidence', threshold=0.9, factor=1)
print(tokenizer.batch_decode(out[0])[0].split("<|endoftext|>")[0])

<|startoftext|><|start_header_id|>user<|end_header_id|>

Given the following question, reason and give a final answer to the question. Your response should end with "The answer is [answer]" where [answer] is the response to the question.

Here are a few examples of questions and answers:

Question: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
Answer: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.

Question: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
Answer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.

Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
Answer: Originally, Leah had 32 chocolates. Her sister had 

In [14]:
sequences = out[0]
        
with torch.inference_mode():
    model_output = model(sequences, output_attentions=True, output_hidden_states=True)
        
logits = model_output.logits.log_softmax(dim=-1).to(torch.float16)
attentions = model_output.attentions
hidden_states = model_output.hidden_states
prompt_length = input_ids.shape[1]      

In [28]:
hidden_states

(tensor([[[ 0.0008,  0.0140,  0.0544,  ..., -0.0294, -0.0302,  0.0206],
          [-0.0062, -0.0071,  0.0132,  ...,  0.0155,  0.0059,  0.0056],
          [-0.1387,  0.1816,  0.0762,  ..., -0.0483, -0.1504,  0.0095],
          ...,
          [ 0.0069, -0.0330,  0.1216,  ...,  0.0461,  0.0286,  0.0072],
          [ 0.0069, -0.0330,  0.1216,  ...,  0.0461,  0.0286,  0.0072],
          [ 0.0069, -0.0330,  0.1216,  ...,  0.0461,  0.0286,  0.0072]]],
        device='cuda:0', dtype=torch.bfloat16),
 tensor([[[ 0.1621, -0.0403,  0.0229,  ..., -0.2412,  0.0035,  0.1426],
          [ 0.0435, -0.0046,  0.0864,  ..., -0.0525,  0.0325,  0.1367],
          [-0.1875,  0.0288,  0.2275,  ...,  0.0413, -0.0596, -0.0229],
          ...,
          [ 0.2637, -0.0654, -0.1758,  ..., -0.2891, -0.0645,  0.2129],
          [ 0.2754, -0.0576, -0.1904,  ..., -0.2930, -0.0674,  0.2266],
          [ 0.2773, -0.0544, -0.1924,  ..., -0.2930, -0.0659,  0.2266]]],
        device='cuda:0', dtype=torch.bfloat16),
 tenso

In [39]:
len(hidden_states), hidden_states[0].shape

(33, torch.Size([1, 1100, 4096]))

In [52]:
def rearrange_hidden_states(hidden_states, prompt_len, token_len):
    """
    hidden_states: tuple of 33 tensors
                   each tensor shape [1, prompt_len + token_len, 4096]

    returns: tuple of length token_len
             each element is a tuple of 33 tensors
    """
    num_layers = len(hidden_states)

    result = []

    for t in range(token_len):
        per_token = []

        for layer in hidden_states:
            if t == 0:
                # prompt tensor only once per token group
                per_token.append(layer[:, :prompt_len, :])   # [1, prompt_len, 4096]
            else:
                per_token.append(layer[:, prompt_len + t - 1 : prompt_len + t, :])
                # [1, 1, 4096]

        result.append(tuple(per_token))

    return tuple(result)

In [70]:
hs = rearrange_hidden_states(hidden_states, input_ids.shape[1], 50)
len(hs), len(hs[0])

(50, 33)

In [71]:
hs[0][0].shape, hs[1][0].shape

(torch.Size([1, 844, 4096]), torch.Size([1, 1, 4096]))

In [72]:
hidden_states[0][0, 844]

tensor([-0.1416,  0.2500, -0.1270,  ...,  0.3730, -0.0698, -0.0369],
       device='cuda:0', dtype=torch.bfloat16)

In [73]:
hs[1][0][0, 0]

tensor([-0.1416,  0.2500, -0.1270,  ...,  0.3730, -0.0698, -0.0369],
       device='cuda:0', dtype=torch.bfloat16)

In [156]:
model

LLaDAModelLM(
  (model): LLaDAModel(
    (transformer): ModuleDict(
      (wte): Embedding(126464, 4096)
      (emb_drop): Dropout(p=0.0, inplace=False)
      (ln_f): RMSLayerNorm()
      (blocks): ModuleList(
        (0-31): 32 x LLaDALlamaBlock(
          (dropout): Dropout(p=0.0, inplace=False)
          (act): SiLU()
          (attn_out): Linear(in_features=4096, out_features=4096, bias=False)
          (ff_out): Linear(in_features=12288, out_features=4096, bias=False)
          (rotary_emb): RotaryEmbedding()
          (attn_norm): RMSLayerNorm()
          (ff_norm): RMSLayerNorm()
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (ff_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
        )
      )
    

In [139]:
stop_sequences = ["<|endoftext|>", "<|eot_id|>"]
stop_token_ids = [tokenizer.encode(stop_seq, add_special_tokens=False) for stop_seq in stop_sequences]

processed_sequences = []
processed_logits = []

for i in range(sequences.shape[0]):
    generated_token_ids = sequences[i, prompt_length:]
    trunc_length = generated_token_ids.shape[0]

    # Find first stop sequence in token ids
    for stop_seq_ids in stop_token_ids:
        if not stop_seq_ids:
            continue
        stop_seq_len = len(stop_seq_ids)
        if stop_seq_len == 0 or stop_seq_len > trunc_length:
            continue
        for idx in range(trunc_length - stop_seq_len + 1):
            if torch.equal(generated_token_ids[idx:idx+stop_seq_len], torch.tensor(stop_seq_ids, device=generated_token_ids.device)):
                trunc_length = idx
                break
        # if found a truncation, break out of stop_seq_ids loop
        if trunc_length != generated_token_ids.shape[0]:
            break

    trimmed_generated = generated_token_ids[:trunc_length]
    n_tokens = trimmed_generated.shape[0]
    processed_sequences.append(torch.cat([input_ids[i], trimmed_generated]))
    processed_logits.append(logits[i, prompt_length:prompt_length+n_tokens, :])       

In [140]:
processed_sequences = torch.stack(processed_sequences, dim=0)
processed_logits = torch.stack(processed_logits, dim=0).transpose(1, 0)
processed_logits = tuple(logit for logit in processed_logits)
generated_answers = tokenizer.batch_decode(processed_sequences, skip_special_tokens=False)
print(generated_answers, n_tokens, prompt_length, processed_sequences.shape)

['<|startoftext|><|start_header_id|>user<|end_header_id|>\n\nGiven the following question, reason and give a final answer to the question. Your response should end with "The answer is [answer]" where [answer] is the response to the question.\n\nHere are a few examples of questions and answers:\n\nQuestion: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\nAnswer: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.\n\nQuestion: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\nAnswer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.\n\nQuestion: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\nAnswer: Originally, Leah had 32 chocolates. 

In [141]:
%%time

attentions_greedy = []
if True:
    for token_idx in range(n_tokens):
        if token_idx == 0:
            attentions_greedy.append(torch.cat([attention[:, :, :prompt_length + token_idx, :prompt_length + token_idx] for attention in attentions]).unsqueeze(-4))
        else:
            attentions_greedy.append(torch.cat([attention[:, :, prompt_length + token_idx - 1, :prompt_length + token_idx] for attention in attentions]).unsqueeze(-3).unsqueeze(-2))

CPU times: user 28.3 ms, sys: 4.41 ms, total: 32.7 ms
Wall time: 32.3 ms


In [153]:
attentions[31][0, 0]

tensor([[9.2697e-04, 1.0431e-05, 5.1117e-04,  ..., 7.7114e-07, 3.0756e-05,
         2.3283e-08],
        [9.9609e-01, 1.0872e-04, 1.5945e-03,  ..., 1.8758e-12, 5.1260e-05,
         7.1668e-10],
        [5.3516e-01, 5.1880e-04, 2.8320e-02,  ..., 3.1851e-07, 2.8320e-02,
         3.1479e-07],
        ...,
        [2.8014e-05, 2.2465e-10, 3.6955e-05,  ..., 2.8014e-05, 7.1875e-01,
         6.0027e-10],
        [7.8125e-03, 8.4192e-07, 5.8746e-04,  ..., 1.8720e-07, 1.3733e-02,
         3.0850e-09],
        [1.8477e-05, 1.5832e-08, 2.9355e-06,  ..., 1.3097e-08, 3.1836e-01,
         3.9673e-04]], device='cuda:0', dtype=torch.bfloat16)

In [142]:
prompt_length, attentions_greedy[0].shape, attentions_greedy[1].shape, attentions_greedy[2].shape

(844,
 torch.Size([32, 1, 32, 844, 844]),
 torch.Size([32, 1, 32, 1, 845]),
 torch.Size([32, 1, 32, 1, 846]))

In [102]:
prompt_length, attentions_greedy[0].shape, attentions_greedy[1].shape, attentions_greedy[2].shape

(844,
 torch.Size([32, 1, 32, 844, 844]),
 torch.Size([32, 1, 32, 1, 845]),
 torch.Size([32, 1, 32, 1, 846]))

In [144]:
attentions_greedy[2][0, 0, 0, 0]

tensor([1.9169e-04, 9.6321e-05, 6.1989e-05, 1.4114e-04, 1.0071e-03, 1.1368e-03,
        1.3161e-04, 4.6730e-04, 9.8705e-05, 2.8801e-04, 2.0294e-03, 1.0109e-04,
        1.6861e-03, 5.8651e-05, 3.1662e-04, 4.2915e-05, 6.3896e-05, 6.1035e-04,
        3.7766e-04, 1.1539e-04, 1.7166e-03, 4.6015e-05, 9.2506e-05, 1.0443e-04,
        9.3937e-05, 1.2054e-03, 9.4414e-05, 1.2398e-04, 5.8174e-05, 8.7738e-04,
        4.9591e-04, 7.0572e-04, 1.0777e-04, 1.2493e-04, 4.9973e-04, 5.6076e-04,
        2.0695e-04, 5.6458e-04, 2.9182e-04, 5.3883e-05, 5.2643e-04, 3.0136e-04,
        1.4591e-04, 1.3046e-03, 7.4768e-04, 6.8665e-04, 5.1737e-05, 1.4973e-04,
        3.2806e-04, 1.2064e-04, 3.7909e-05, 1.3885e-03, 4.3392e-05, 1.6632e-03,
        3.5524e-05, 4.3488e-04, 1.0986e-03, 1.1368e-03, 3.9101e-04, 4.0245e-04,
        3.0708e-04, 3.1471e-04, 1.0986e-03, 7.2098e-04, 1.2970e-04, 3.2187e-05,
        2.2030e-04, 3.0136e-04, 1.1539e-04, 8.3542e-04, 1.8501e-04, 4.6015e-05,
        8.2493e-05, 6.8665e-05, 6.7234e-

In [19]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"  # adjust to the exact repo you have access to

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (device == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16

tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model_ = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map=None,
).to(device)
model.eval()

prompt = text

inputs = tok(prompt, return_tensors="pt").to(device)

In [20]:
with torch.no_grad():
    gen_out = model_.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        top_k=50,
        return_dict_in_generate=True,
        output_attentions=True,     # <- attention tensors
        output_hidden_states=True,     # <- attention tensors
        output_scores=True,         # optional: per-step logits
        use_cache=True,
    )

# Generated tokens
gen_ids = gen_out.sequences  # [batch, prompt_len + new_len]
text = tok.decode(gen_ids[0], skip_special_tokens=True)

# Attentions:
# gen_out.attentions is typically a tuple over generation steps,
# each element is a tuple over layers, each layer is [B, H, Q, K]
attentions = gen_out.attentions

print(text)
print("num_steps:", len(attentions))
print("num_layers:", len(attentions[0]))
print("one tensor shape (step0, layer0):", attentions[0][0].shape)

# Example: last-token attention at the final step for the last layer
last_step = len(attentions) - 1
last_layer = len(attentions[last_step]) - 1
last_attn = attentions[last_step][last_layer]          # [B, H, Q, K]
last_token_attn = last_attn[:, :, -1, :]               # [B, H, K]
print("last_token_attn shape:", last_token_attn.shape)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
`torch.nn.functional.scaled_dot_product_attention` does not support `output_attentions=True`. Falling back to eager attention. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


Given the following question, reason and give a final answer to the question. Your response should end with "The answer is [answer]" where [answer] is the response to the question.

Here are a few examples of questions and answers:

Question: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
Answer: There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The answer is 6.

Question: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
Answer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The answer is 5.

Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
Answer: Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, t

In [23]:
len(gen_out.hidden_states)

5

In [25]:
len(gen_out.hidden_states[0])

17

In [26]:
gen_out.hidden_states[0][0].shape

torch.Size([1, 765, 2048])

In [27]:
gen_out.hidden_states[1][0].shape

torch.Size([1, 1, 2048])

In [112]:
forwardpass_attentions = model_(
    gen_ids, output_attentions=True
).attentions

In [113]:
forwardpass_attentions[-1][-1, -1, -2][-6:-1]

tensor([0.0002, 0.0018, 0.0032, 0.0293, 0.0060], device='cuda:0',
       dtype=torch.bfloat16, grad_fn=<SliceBackward0>)

In [114]:
forwardpass_attentions[0].shape

torch.Size([1, 32, 770, 770])

In [115]:
attentions[-1][-1][-1, -1, -1][-5:]

tensor([0.0002, 0.0017, 0.0033, 0.0297, 0.0061], device='cuda:0',
       dtype=torch.bfloat16)

In [116]:
attentions[0][0].shape

torch.Size([1, 32, 765, 765])

In [117]:
torch.stack(attentions[0]).shape, torch.stack(attentions[1]).shape, torch.stack(attentions[2]).shape

(torch.Size([16, 1, 32, 765, 765]),
 torch.Size([16, 1, 32, 1, 766]),
 torch.Size([16, 1, 32, 1, 767]))

In [152]:
len_tokens = inputs["input_ids"].shape[1]
token_i = 3

In [153]:
attentions[token_i][-1][-1, -1, -1][-token_i:]

tensor([0.0017, 0.0025, 0.0280], device='cuda:0', dtype=torch.bfloat16)

In [154]:
forwardpass_attentions[-1][-1, -1, len_tokens + token_i - 1, len_tokens : len_tokens + token_i]

tensor([0.0017, 0.0025, 0.0281], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<SliceBackward0>)

In [134]:
attentions[-1][-1].shape

torch.Size([1, 32, 1, 769])

In [136]:
forwardpass_attentions[-1].shape

torch.Size([1, 32, 770, 770])

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Efficient-Large-Model/Fast_dLLM_7B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

# Fast-dLLM v2 parallel decoding
gen_ids = model.generate(
    inputs["input_ids"],
    tokenizer=tokenizer,
    max_new_tokens=512,
    small_block_size=8,
    threshold=0.9,
)

response = tokenizer.decode(
    gen_ids[0][inputs["input_ids"].shape[1]:], 
    skip_special_tokens=True
)
print(response)

A new version of the following files was downloaded from https://huggingface.co/Efficient-Large-Model/Fast_dLLM_7B:
- configuration.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


ImportError: cannot import name 'layer_type_validation' from 'transformers.configuration_utils' (/home/jovyan/.mlspace/envs/d-llm/lib/python3.11/site-packages/transformers/configuration_utils.py)

In [120]:
# pip install accelerate
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B-Instruct",
    device_map="auto",
    attn_implementation="eager",
)

In [65]:
prompt = lambda t: [{"role": "assistant", "content": t}]
t = "MOSCOW IS THE CAPITAL OF RUSSIA"
text = tokenizer.apply_chat_template(
    prompt(t),
    tokenize=False,
    add_generation_prompt=False
)
print(text)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 25 Aug 2025

<|eot_id|><|start_header_id|>assistant<|end_header_id|>

MOSCOW IS THE CAPITAL OF RUSSIA<|eot_id|>


In [121]:
if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token

In [122]:
orig_inputs = tokenizer(
    t,
    return_tensors="pt",
    padding=True,
    truncation=True,
    add_special_tokens=False
).to("cuda")  
orig_inputs['input_ids']

tensor([[   44, 80061,  3387,  3507,  3247, 78387,  3083, 69925,  1242,  5987]],
       device='cuda:0')

In [123]:
tokenizer.decode(orig_inputs['input_ids'][0])

'MOSCOW IS THE CAPITAL OF RUSSIA'

In [124]:
inputs = tokenizer(
    text,
    return_tensors="pt",
    padding=True,
    truncation=True
).to("cuda")  
inputs['input_ids']

tensor([[128000, 128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,
           2696,     25,   6790,    220,   2366,     18,    198,  15724,   2696,
             25,    220,    914,   5033,    220,   2366,     20,    271, 128009,
         128006,  78191, 128007,    271,     44,  80061,   3387,   3507,   3247,
          78387,   3083,  69925,   1242,   5987, 128009]], device='cuda:0')

In [129]:
with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

In [130]:
attn_mask = tuple(
    attention.to("cpu") for attention in outputs.attentions
)
attn_mask = (
    torch.cat(attn_mask).float().numpy()
)

In [146]:
n = orig_inputs['input_ids'].shape[1]
attn_mask[0, 0, :-1, :-1][-n-1:-1, -n:]

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.00459691, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.05565343, 0.02467631, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.07823236, 0.03404632, 0.01831218, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.10983487, 0.06980401, 0.05484834, 0.02446768, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.06174549, 0.07671255, 0.12358954, 0.1236627 , 0.05088706,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.0210875 , 0.02120145, 0.08454438, 0.12001671, 0.11873778,
        0.05244074, 0.        , 0.        , 0.        , 0.        ],
       [0.01265413, 0.01392625, 0.0560983

In [141]:
attn_mask[0, 0, :-1, :-1][-n:-1, -n+1:]

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.02467631, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.03404632, 0.01831218, 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.06980401, 0.05484834, 0.02446768, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.07671255, 0.12358954, 0.1236627 , 0.05088706, 0.        ,
        0.        , 0.        , 0.        , 0.        ],
       [0.02120145, 0.08454438, 0.12001671, 0.11873778, 0.05244074,
        0.        , 0.        , 0.        , 0.        ],
       [0.01392625, 0.0560983 , 0.22160676, 0.45993677, 0.09378768,
        0.01597569, 0.        , 0.        , 0.        ],
       [0.01244297, 0.03723636, 0.06890333, 0.1118725 , 0.13830835,
        0.11859953, 0.02777595, 0.        , 0.        ],


In [143]:
attn_mask[0, 0, :-1, :-1][-2]

array([0.21253145, 0.20894064, 0.00059876, 0.00068251, 0.00085378,
       0.00129856, 0.00234491, 0.00090043, 0.00199258, 0.00202731,
       0.0040028 , 0.00206696, 0.00135797, 0.00036415, 0.00030717,
       0.00106049, 0.00215468, 0.00234566, 0.00352885, 0.00538402,
       0.00191742, 0.00252668, 0.00336486, 0.00233047, 0.00204469,
       0.01167798, 0.01621803, 0.00591684, 0.01172849, 0.00904555,
       0.03419263, 0.01214368, 0.0123238 , 0.02578776, 0.03307661,
       0.04646458, 0.11057573, 0.11736707, 0.06823494, 0.01831855,
       0.        ], dtype=float32)

In [126]:
outputs.logits.shape

torch.Size([1, 42, 128256])

In [127]:
inputs['input_ids'].shape

torch.Size([1, 42])

In [128]:
outputs.logits[0, 39].argsort()

tensor([ 6960,  1617,   336,  ..., 87582, 22774,  5987], device='cuda:0')

In [95]:
inputs["input_ids"][0, 40]

tensor(5987, device='cuda:0')

In [98]:
inputs["input_ids"][0, -orig_inputs['input_ids'].shape[1]-1:-1]

tensor([   44, 80061,  3387,  3507,  3247, 78387,  3083, 69925,  1242,  5987],
       device='cuda:0')

In [112]:
logits = outputs.logits[0, :-1]
for i, l in zip(inputs["input_ids"][0][1:], logits):
    print(i, l.argsort()[-3:])

tensor(128000, device='cuda:0') tensor([  791,     2, 16309], device='cuda:0')
tensor(128006, device='cuda:0') tensor([  791,     2, 16309], device='cuda:0')
tensor(9125, device='cuda:0') tensor([   198,  78191, 128006], device='cuda:0')
tensor(128007, device='cuda:0') tensor([2953,  882,  198], device='cuda:0')
tensor(271, device='cuda:0') tensor([  198,   271, 78191], device='cuda:0')
tensor(38766, device='cuda:0') tensor([791,  32, 567], device='cuda:0')
tensor(1303, device='cuda:0') tensor([1022,  279, 1303], device='cuda:0')
tensor(33025, device='cuda:0') tensor([10564,  6964,   311], device='cuda:0')
tensor(2696, device='cuda:0') tensor([10564, 22241, 12299], device='cuda:0')
tensor(25, device='cuda:0') tensor([ 1473, 16842,    25], device='cuda:0')
tensor(6790, device='cuda:0') tensor([1721, 6186,  220], device='cuda:0')
tensor(220, device='cuda:0') tensor([2437, 1721,  220], device='cuda:0')
tensor(2366, device='cuda:0') tensor([1721, 2366,   16], device='cuda:0')
tensor(18, de

In [118]:
log_probs_orig = logits[-orig_inputs['input_ids'].shape[1]-1:-1]
for i, l in zip(orig_inputs['input_ids'][0], log_probs_orig):
    print(i, l.argsort()[-3:])

tensor(44, device='cuda:0') tensor([ 791,   40, 6854], device='cuda:0')
tensor(80061, device='cuda:0') tensor([ 380,  371, 6430], device='cuda:0')
tensor(3387, device='cuda:0') tensor([ 3143, 40057,  3387], device='cuda:0')
tensor(3507, device='cuda:0') tensor([88030,   320,    11], device='cuda:0')
tensor(3247, device='cuda:0') tensor([ 2006, 57318,   362], device='cuda:0')
tensor(78387, device='cuda:0') tensor([18880,  6864, 78387], device='cuda:0')
tensor(3083, device='cuda:0') tensor([ 3651, 47652,  3083], device='cuda:0')
tensor(69925, device='cuda:0') tensor([ 8524,   432, 69925], device='cuda:0')
tensor(1242, device='cuda:0') tensor([76439,   784,  1242], device='cuda:0')
tensor(5987, device='cuda:0') tensor([87582, 22774,  5987], device='cuda:0')


In [119]:
log_probs_orig.shape, orig_inputs['input_ids'][0].shape

(torch.Size([10, 128256]), torch.Size([10]))

In [12]:
path = "./workdir/output_dllm_dec25/qa/GSAI-ML/LLaDA-1.5/['gsm8k', 'main']/2025-12-15/11-26-29/ue_manager_seed1"

data = torch.load(path, weights_only=False)
data["stats"]["greedy_texts"]

['Peter wants to exercise for twice the amount of time he did on Monday and Sunday combined. First, we need to find the total time he exercised on Monday and Sunday. \n\nOn Monday, Peter exercised for 16 minutes. On Sunday, he exercised for 23 minutes. So, the total time for Monday and Sunday is:\n\\[ 16 + 23 = 39 \\text{ minutes} \\]\n\nPeter wants to exercise for twice this amount on Tuesday:\n\\[ 2 \\times 39 = 78 \\text{ minutes} \\]\n\nTherefore, Peter needs to exercise for 78 minutes on Tuesday to reach his goal. The answer is 78.',
 'To determine how many pieces of paper are used in a 32-page tabloid, we need to consider the structure of the tabloid. Each pageloid consists of two pages: one on the left side and one on the right side.\n\nFor a 32-page tabloid:\n- The left side has pages 1 through 16.\n- The right side has pages 17 through 32.\n\nSince each pageloid is made by folding a piece of paper vertically, we need to determine how many pieces of paper are required to produc

In [26]:
ln = len(best_tokens)
best_tokens[ln - n_alternatives : ln]

array([3, 6, 1, 2, 4])

In [27]:
best_tokens

array([5, 9, 0, 7, 8, 3, 6, 1, 2, 4])

In [30]:
lt[np.argsort(lt)]

array([-1.07806805, -0.84516091, -0.79086983, -0.74725146, -0.61941113,
       -0.60009439, -0.52913428,  0.10576243,  0.45367639,  0.99767519])

In [32]:
lt[:-1].shape

(9,)

In [3]:
from lm_polygraph.generation_metrics.alignscore import AlignScore
from lm_polygraph.ue_metrics.ue_metric import get_random_scores
from tqdm import tqdm

scorer_sum = AlignScore(target_is_claims=False)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
from lm_polygraph.ue_metrics.pred_rej_area import PredictionRejectionArea

prr = PredictionRejectionArea(0.5)

In [5]:
import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel


def forward_process(batch, prompt_index, mask_id):
    b, l = batch.shape

    target_len = (l - prompt_index.sum()).item()
    k = torch.randint(1, target_len + 1, (), device=batch.device)

    x = torch.round(torch.linspace(float(k), k + (b - 1) * (target_len / b), steps=b, device=batch.device)).long()
    x = ((x - 1) % target_len) + 1
    assert x.min() >= 1 and x.max() <= target_len

    indices = torch.arange(target_len, device=batch.device).repeat(b, 1)
    is_mask = indices < x.unsqueeze(1)
    for i in range(b):
        is_mask[i] = is_mask[i][torch.randperm(target_len)]

    is_mask = torch.cat((torch.zeros(b, prompt_index.sum(), dtype=torch.bool, device=batch.device), is_mask), dim=1)
    noisy_batch = torch.where(is_mask, mask_id, batch)

    # Return the masked batch and the mask ratio
    return noisy_batch, (x / target_len).unsqueeze(1).repeat(1, l)


def get_logits(model, batch, prompt_index, cfg_scale, mask_id):
    if cfg_scale > 0.:
        assert len(prompt_index) == batch.shape[1]
        prompt_index = prompt_index.unsqueeze(0).repeat(batch.shape[0], 1)
        un_batch = batch.clone()
        un_batch[prompt_index] = mask_id
        batch = torch.cat([batch, un_batch])

    input = batch
    logits = model(input).logits

    if cfg_scale > 0.:
        logits, un_logits = torch.chunk(logits, 2, dim=0)
        logits = un_logits + (cfg_scale + 1) * (logits - un_logits)
    return logits


@ torch.no_grad()
def get_log_likelihood(model, prompt, answer, mc_num=128, batch_size=16, cfg_scale=0., mask_id=126336):
    '''
    Args:
        model: Mask predictor.
        prompt: A tensor of shape (l1).
        answer: A tensor of shape (l2).
        mc_num: Monte Carlo estimation times.
                As detailed in Appendix B.5. Since MMLU, CMMLU, and C-EVAL only require the likelihood of a single token, a
                single Monte Carlo estimate is sufficient for these benchmarks. For all other benchmarks, we find that 128
                Monte Carlo samples are adequate to produce stable results.
        batch_size: Mini batch size.
        cfg_scale: Unsupervised classifier-free guidance scale.
        mask_id: The toke id of [MASK] is 126336.
    '''
    seq = torch.concatenate([prompt, answer])[None, :]
    seq = seq.repeat((batch_size, 1)).to(model.device)
    prompt_index = torch.arange(seq.shape[1], device=model.device) < len(prompt)

    loss_ = []
    for _ in range(mc_num // batch_size):
        perturbed_seq, p_mask = forward_process(seq, prompt_index, mask_id)
        mask_index = perturbed_seq == mask_id

        logits = get_logits(model, perturbed_seq, prompt_index, cfg_scale, mask_id)

        loss = F.cross_entropy(logits[mask_index], seq[mask_index], reduction='none') / p_mask[mask_index]
        loss = loss.sum() / batch_size

        loss_.append(loss.item())

    return - sum(loss_) / len(loss_)

In [6]:
device = 'cuda'

llada_model = AutoModel.from_pretrained('GSAI-ML/LLaDA-1.5', trust_remote_code=True, torch_dtype=torch.bfloat16, device_map="auto").eval()
tokenizer = AutoTokenizer.from_pretrained('GSAI-ML/LLaDA-1.5', trust_remote_code=True)

The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.
Loading checkpoint shards: 100%|██████████| 6/6 [00:02<00:00,  2.00it/s]


In [8]:
import re
import torch
from sklearn.metrics import roc_auc_score
import string

def preproc_method_name(x):
    return x

def get_table_short(path, metric_names=["Rouge_rougeL", 'AlignScore'], dataset="CoQA", ue_metric="prr_0.5_normalized", update_alignscore_sum=False,
              update_alignscore=False, aggregated=False, return_score=False, level="sequence", upd=False, roc_auc=False, upd_med=False, add_nll=False, mc_num=16):
    
    man = torch.load(path, weights_only=False)
    df = {}
    
    methods = np.unique([x[1] for x in man['metrics'].keys() if "2side" not in x[1]])
    if add_nll:
        methods = np.array(list(methods) + ["SampleNLL"])
        
        greedy_text = np.array(man["stats"][("greedy_texts")])
        input_text = np.array(man["stats"][("input_texts")])

        ue = []
        for i in tqdm(range(len(greedy_text))):
            prompt = torch.tensor(tokenizer(input_text[i])['input_ids']).to(llada_model.device)
            answer = torch.tensor(tokenizer(greedy_text[i])['input_ids']).to(llada_model.device)
            if len(answer):
                ue.append(-get_log_likelihood(llada_model, prompt, answer, mc_num=mc_num))
            else: 
                ue.append(1e+3)
        sample_nll = np.array(ue)
    
    if update_alignscore_sum and (dataset in ['XSUM', 'SamSum', 'CNN']):
        greedy_texts = [x if len(x.strip())>0 else "-" for x in man['stats']["greedy_texts"]]
        greedy_texts = np.array(greedy_texts)
        target_texts = man['stats']["target_texts"]
        input_texts = man['stats']["input_texts"]
        if "Summary (one sentence):" in input_texts[0]:
            context = np.array([x.split("Text:")[-1].split("Summary (one sentence):")[0] for x in input_texts])
        elif "End of dialogue" in input_texts[0]:
            context = np.array([x.split("Here\'s the dialogue:\n\n")[-1].split(" (End of dialogue)")[0] for x in input_texts])
        elif "End of text" in input_texts[0]:
            context = np.array([x.split("Here\'s the text:\n\n")[-1].split(" (End of text)")[0] for x in input_texts])
        alignscores_0 = scorer_sum({"greedy_texts": greedy_texts, "target_texts": context}, context)

    if (dataset in ['GSM8k']):
        greedy_texts = [x if len(x.strip())>0 else "-" for x in man['stats']["greedy_texts"]]
        greedy_texts = np.array(greedy_texts)
        target_texts = np.array(man['stats']["target_texts"])
        acc = np.array(man["gen_metrics"][("sequence", "Accuracy")])

        answers = []
        for text in greedy_texts[acc == 0]:
            answer_match = re.search(
                r'(?:The answer is|Answer[:]?)[^\d]*(\d+)',
                text, 
                flags=re.IGNORECASE
            )
            try:
                answer = answer_match.group(1).strip()
                answer = answer.translate(str.maketrans("", "", string.punctuation))
                answers.append(int(answer))
            except:
                answers.append(-1e10) 
        answers = np.array(answers)
        targets = []
        for text in target_texts[acc == 0]:
            output_ignore_regex = re.compile("(?s).*#### ")
            answer = output_ignore_regex.sub("", text)
            answer = answer.translate(str.maketrans("", "", string.punctuation))
            targets.append(int(answer)) 
        targets = np.array(targets)

        acc[acc == 0] = (answers == targets) * 1

        
    for method in methods:
        for metric in metric_names:
            key = (level, method, metric, ue_metric)
            scores = df.get(method, [])
            if method == "SampleNLL":
                ue = sample_nll
                if update_alignscore_sum and (dataset in ['XSUM', 'SamSum', 'CNN']):
                    metrics = alignscores_0
                elif (dataset in ['GSM8k']) and (metric == "Accuracy"):
                    metrics = acc
                else:
                    metrics = np.array(man['gen_metrics'][(level, metric)])
                mean_val = prr(ue, metrics) 
                oracle = prr(-metrics, metrics)
                random = get_random_scores(prr, metrics)
                scores.append((mean_val - random) / (oracle - random))

            elif update_alignscore_sum and (dataset in ['XSUM', 'SamSum', 'CNN']):
                metrics = np.array(man['gen_metrics'][(level, metric)])
                ue = np.array([x if not isinstance(x, complex) else x.real for x in man['estimations'][(level, method)]])
                metrics = alignscores_0
                mean_val = prr(ue, metrics) 
                oracle = prr(-metrics, metrics)
                random = get_random_scores(prr, metrics)
                scores.append((mean_val - random) / (oracle - random))
            elif (dataset in ['GSM8k']) and (metric == "Accuracy"):
                metrics = acc
                ue = np.array([x if not isinstance(x, complex) else x.real for x in man['estimations'][(level, method)]])
                mean_val = prr(ue, metrics) 
                oracle = prr(-metrics, metrics)
                random = get_random_scores(prr, metrics)
                scores.append((mean_val - random) / (oracle - random))
            else:
                scores.append(man['metrics'][key])
            df[method] = scores
    for metric in metric_names:
        if update_alignscore_sum and (dataset in ['XSUM', 'SamSum', 'CNN']):
            print(f"{metric}: ", np.mean(alignscores_0))
        elif (dataset in ['GSM8k']) and (metric == "Accuracy"):
            print(f"{metric}: ", np.mean(acc))
        else:
            print(f"{metric}: ", np.mean(man['gen_metrics'][("sequence", metric)]))
    df0 = pd.DataFrame(df, index=pd.MultiIndex.from_tuples([(dataset, m) for m in metric_names])).T
    df0.loc[:, (dataset, 'Method')] = [x for x in df0.index]
    return df0.reset_index(drop=True)

ERROR! Session/line number was not unique in database. History logging moved to new session 1557


In [12]:
import os
import copy
from scipy.stats import rankdata

base_path = "./workdir/output"
base_path = "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final"

datasets = ['ats/xsum', 'ats/samsum', "ats/['cnn_dailymail', '3.0.0']", "nmt/['wmt14', 'fr-en']", "nmt/['wmt19', 'de-en']", "qa/['truthful_qa', 'generation']", 'qa/coqa', 'qa/sciq', "qa/['trivia_qa', 'rc.nocontext']", "qa/['cais/mmlu', 'all']", "qa/['gsm8k', 'main']"]
datasets_names = ['XSUM', 'SamSum', 'CNN', "WMT14", "WMT19", "TruthfulQA", 'CoQA', 'SciQ', "TriviaQA", "MMLU", "GSM8k"]

datasets = ['ats/samsum', "nmt/['wmt14', 'fr-en']", "qa/['trivia_qa', 'rc.nocontext']"]
datasets_names = ['SamSum', "WMT14", "TriviaQA"]

models = ['GSAI-ML/LLaDA-1.5', 'meta-llama/Llama-3.1-8B', 'meta-llama/Llama-3.1-8B-Instruct']

paths = {
    "GSAI-ML/LLaDA-1.5": {
        "GSM8k": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/['gsm8k', 'main']/2025-06-26/15-22-42/ue_manager_seed1",
        "CNN": "./workdir/output_dllm/ats/GSAI-ML/LLaDA-1.5/['cnn_dailymail', '3.0.0']/2025-06-28/08-10-10/ue_manager_seed1",
        "XSUM": "./workdir/output_dllm/ats/GSAI-ML/LLaDA-1.5/xsum/2025-06-29/14-26-07/ue_manager_seed1",
        # "SamSum": "./workdir/output_dllm/ats/GSAI-ML/LLaDA-1.5/samsum/2025-06-29/14-26-07/ue_manager_seed1",
        "SamSum": "./workdir/output_dllm_dec25/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2025-12-18/00-27-22/ue_manager_seed1",
        "CoQA": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/coqa/2025-06-29/09-42-26/ue_manager_seed1",
        "SciQ": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/sciq/2025-06-30/07-41-34/ue_manager_seed1",
        "TruthfulQA": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/['truthful_qa', 'generation']/2025-06-30/07-41-34/ue_manager_seed1",
        # "WMT14": "./workdir/output_dllm/nmt/GSAI-ML/LLaDA-1.5/['wmt14', 'fr-en']/2025-06-30/11-50-18/ue_manager_seed1",
        "WMT14": "./workdir/output_dllm_dec25/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2025-12-16/16-35-15/ue_manager_seed1",
        "WMT19": "./workdir/output_dllm/nmt/GSAI-ML/LLaDA-1.5/['wmt19', 'de-en']/2025-06-30/11-50-18/ue_manager_seed1",
        # "TriviaQA": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/['trivia_qa', 'rc.nocontext']/2025-06-30/20-09-50/ue_manager_seed1",
        "TriviaQA": "./workdir/output_dllm_dec25/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2025-12-17/16-31-56/ue_manager_seed1",
        "MMLU": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/['cais/mmlu', 'all']/2025-06-30/20-09-50/ue_manager_seed1",
    },
    "meta-llama/Llama-3.1-8B-Instruct": {
        "WMT19": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/nmt/meta-llama/Llama-3.1-8B-Instruct/['wmt19', 'de-en']/2025-06-11/19-38-21/ue_manager_seed1",
        "GSM8k": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/qa/meta-llama/Llama-3.1-8B-Instruct/['gsm8k', 'main']/2025-06-11/12-23-57/ue_manager_seed1",
        "CoQA": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/qa/meta-llama/Llama-3.1-8B-Instruct/coqa/2025-06-01/11-51-41/ue_manager_seed1",
        "TriviaQA": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/qa/meta-llama/Llama-3.1-8B-Instruct/['trivia_qa', 'rc.nocontext']/2025-06-02/10-08-42/ue_manager_seed1",
        "SamSum": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/ats/meta-llama/Llama-3.1-8B-Instruct/samsum/2025-06-01/16-59-58/ue_manager_seed1",
        "WMT14": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/nmt/meta-llama/Llama-3.1-8B-Instruct/['wmt14', 'fr-en']/2025-06-01/09-39-16/ue_manager_seed1",
        "SciQ": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/qa/meta-llama/Llama-3.1-8B-Instruct/sciq/2025-05-31/17-55-02/ue_manager_seed1",
        "TruthfulQA": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/qa/meta-llama/Llama-3.1-8B-Instruct/['truthful_qa', 'generation']/2025-06-18/14-52-07/ue_manager_seed1",
        "MMLU": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/qa/meta-llama/Llama-3.1-8B-Instruct/['cais/mmlu', 'all']/2025-06-01/09-38-45/ue_manager_seed1",
        "MedQUAD": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/qa/meta-llama/Llama-3.1-8B-Instruct/keivalya/MedQuad-MedicalQnADataset/2025-06-18/14-11-14/ue_manager_seed1",
        "XSUM": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/ats/meta-llama/Llama-3.1-8B-Instruct/xsum/2025-06-20/18-37-07/ue_manager_seed1",
        "CNN": "/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/ats/meta-llama/Llama-3.1-8B-Instruct/['cnn_dailymail', '3.0.0']/2025-06-19/22-18-02/ue_manager_seed1",
    }
}

filename = "ue_manager_seed1"
dfs = {}

all_paths = []
for model in models:
    dfs[model] = {}
    for ds, ds_name in zip(datasets, datasets_names):
        task = ds.split("/")[0]
        ds = '/'.join(ds.split("/")[1:])
        path = f"{base_path}/{task}/{model}/{ds}/"
        final_path = None
        if model in paths.keys() and ds_name in paths[model].keys():
            final_path = paths[model][ds_name]
        else:
            for address, dirs, files in os.walk(path):
                for name in files:
                    full_path = os.path.join(address, name)
                    if full_path.endswith(filename):
                        final_path = copy.deepcopy(full_path)
        if final_path is not None:
            print(final_path)
            all_paths.append(final_path)
            if task == "ats":
                metric_names=['AlignScore']
            elif (task == "qa") and (ds_name in ["MMLU", "GSM8k"]):
                metric_names=['Accuracy']
            elif task == "qa":
                metric_names=["AlignScore"]
            elif task == "nmt":
                metric_names=['Comet']
            man = torch.load(final_path, weights_only=False)

            if ds_name in ['XSUM', 'SamSum', 'CNN']:
                greedy_texts = [x if len(x.strip())>0 else "-" for x in man['stats']["greedy_texts"]]
                greedy_texts = np.array(greedy_texts)
                target_texts = man['stats']["target_texts"]
                input_texts = man['stats']["input_texts"]
                if "Summary (one sentence):" in input_texts[0]:
                    context = np.array([x.split("Text:")[-1].split("Summary (one sentence):")[0] for x in input_texts])
                elif "End of dialogue" in input_texts[0]:
                    context = np.array([x.split("Here\'s the dialogue:\n\n")[-1].split(" (End of dialogue)")[0] for x in input_texts])
                elif "End of text" in input_texts[0]:
                    context = np.array([x.split("Here\'s the text:\n\n")[-1].split(" (End of text)")[0] for x in input_texts])
                score = scorer_sum({"greedy_texts": greedy_texts, "target_texts": context}, context)
            else:
                score = man["gen_metrics"][("sequence", metric_names[0])]
            dfs[model][ds_name] = [np.mean(score)]

./workdir/output_dllm_dec25/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2025-12-18/00-27-22/ue_manager_seed1
./workdir/output_dllm_dec25/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2025-12-16/16-35-15/ue_manager_seed1
./workdir/output_dllm_dec25/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2025-12-17/16-31-56/ue_manager_seed1
/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final/ats/meta-llama/Llama-3.1-8B/samsum/2025-05-05/23-40-26/ue_manager_seed1
/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final/nmt/meta-llama/Llama-3.1-8B/['wmt14', 'fr-en']/2025-05-06/09-38-12/ue_manager_seed1
/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final/qa/meta-llama/Llama-3.1-8B/['trivia_qa', 'rc.nocontext']/2025-05-05/19-12-42/ue_manager_seed1
/home/jovyan/artemvazhentsev/llm-tad-uncertainty/workdir/output_unsup_may25_final_instruct_/ats/meta-llama/Ll

In [13]:
qual = pd.concat([pd.DataFrame(dfs[model], index=[model]) for model in models])
qual.loc[:, "Mean"] = qual[qual.columns].mean(axis=1)
qual

,SamSum,WMT14,TriviaQA,Mean
GSAI-ML/LLaDA-1.5,0.899482,0.859268,0.459859,0.739536
meta-llama/Llama-3.1-8B,0.773345,0.862830,0.685978,0.774051
meta-llama/Llama-3.1-8B-Instruct,0.899392,0.868607,0.726541,0.831514


In [ ]:
import os
import copy
from scipy.stats import rankdata

base_path = "./workdir/output"

datasets = ['ats/xsum', 'ats/samsum', "ats/['cnn_dailymail', '3.0.0']", "nmt/['wmt14', 'fr-en']", "nmt/['wmt19', 'de-en']", "qa/keivalya/MedQuad-MedicalQnADataset", "qa/['truthful_qa', 'generation']", 'qa/coqa', 'qa/sciq', "qa/['trivia_qa', 'rc.nocontext']", "qa/['cais/mmlu', 'all']", "qa/['gsm8k', 'main']"]
datasets_names = ['XSUM', 'SamSum', 'CNN', "WMT14", "WMT19", "MedQUAD", "TruthfulQA", 'CoQA', 'SciQ', "TriviaQA", "MMLU", "GSM8k"]

# datasets = ['ats/xsum', 'ats/samsum', "ats/['cnn_dailymail', '3.0.0']", "nmt/['wmt19', 'de-en']", 'qa/coqa', 'qa/sciq', "qa/['gsm8k', 'main']", "qa/['truthful_qa', 'generation']", "qa/['cais/mmlu', 'all']"]
# datasets_names = ['XSUM', 'SamSum', 'CNN', "WMT19", 'CoQA', "SciQ", "GSM8k", "TruthfulQA", "MMLU"]

# datasets = ["nmt/['wmt14', 'fr-en']"]
# datasets_names = ["WMT14"]

# datasets = ["qa/['trivia_qa', 'rc.nocontext']"]
# datasets_names = ["TriviaQA"]

datasets = ['ats/samsum', "nmt/['wmt14', 'fr-en']", "qa/['trivia_qa', 'rc.nocontext']"]
datasets_names = ['SamSum', "WMT14", "TriviaQA"]

models = ['GSAI-ML/LLaDA-1.5/']

paths = {
    "GSAI-ML/LLaDA-1.5/": {
        # "WMT19": "./workdir/output/nmt/GSAI-ML/LLaDA-1.5/['wmt19', 'de-en']/2025-06-22/11-33-11/ue_manager_seed1",
        # "CoQA": "./workdir/output/qa/GSAI-ML/LLaDA-1.5/['LM-polygraph/coqa', 'continuation']/2025-06-20/20-54-52/ue_manager_seed1",
        # "GSM8k": "./workdir/output/qa/GSAI-ML/LLaDA-1.5/['gsm8k', 'main']/2025-06-22/13-05-43/ue_manager_seed1",
        # "TruthfulQA": "./workdir/output/qa/GSAI-ML/LLaDA-1.5/['truthful_qa', 'generation']/2025-06-23/10-06-41/ue_manager_seed1",
        # "MMLU": "./workdir/output/qa/GSAI-ML/LLaDA-1.5/['cais/mmlu', 'all']/2025-06-23/16-05-45/ue_manager_seed1",
        # "SamSum": "./workdir/output/ats/GSAI-ML/LLaDA-1.5/samsum/2025-06-23/20-37-07/ue_manager_seed1",
        # "XSUM": "./workdir/output/ats/GSAI-ML/LLaDA-1.5/xsum/2025-06-23/19-07-50/ue_manager_seed1",
        ###########################################
        "GSM8k": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/['gsm8k', 'main']/2025-06-26/15-22-42/ue_manager_seed1",
        # "GSM8k": "./workdir/output/qa/GSAI-ML/LLaDA-8B-Instruct/['gsm8k', 'main']/2025-12-11/21-13-27/ue_manager_seed1", 
        "CNN": "./workdir/output_dllm/ats/GSAI-ML/LLaDA-1.5/['cnn_dailymail', '3.0.0']/2025-06-28/08-10-10/ue_manager_seed1",
        "XSUM": "./workdir/output_dllm/ats/GSAI-ML/LLaDA-1.5/xsum/2025-06-29/14-26-07/ue_manager_seed1",
        # "SamSum": "./workdir/output_dllm/ats/GSAI-ML/LLaDA-1.5/samsum/2025-06-29/14-26-07/ue_manager_seed1",
        "SamSum": "./workdir/output_dllm_dec25/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2025-12-18/00-27-22/ue_manager_seed1",
        "CoQA": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/coqa/2025-06-29/09-42-26/ue_manager_seed1",
        "SciQ": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/sciq/2025-06-30/07-41-34/ue_manager_seed1",
        "TruthfulQA": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/['truthful_qa', 'generation']/2025-06-30/07-41-34/ue_manager_seed1",
        "WMT14": "./workdir/output_dllm/nmt/GSAI-ML/LLaDA-1.5/['wmt14', 'fr-en']/2025-06-30/11-50-18/ue_manager_seed1",
        # "WMT14": "./workdir/output_dllm_dec25/nmt/GSAI-ML/LLaDA-1.5/['wmt14', 'fr-en']/2025-12-13/06-16-42/ue_manager_seed1",
        "WMT14": "./workdir/output_dllm_dec25/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2025-12-16/16-35-15/ue_manager_seed1",
        "WMT19": "./workdir/output_dllm/nmt/GSAI-ML/LLaDA-1.5/['wmt19', 'de-en']/2025-06-30/11-50-18/ue_manager_seed1",
        # "TriviaQA": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/['trivia_qa', 'rc.nocontext']/2025-06-30/20-09-50/ue_manager_seed1",
        "TriviaQA": "./workdir/output_dllm_dec25/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2025-12-17/16-31-56/ue_manager_seed1",
        "MMLU": "./workdir/output_dllm/qa/GSAI-ML/LLaDA-1.5/['cais/mmlu', 'all']/2025-06-30/20-09-50/ue_manager_seed1",
    }
}

filename = "ue_manager_seed1"
dfs = {}

all_paths = []
for model in models:
    dfs[model] = {}
    for ds, ds_name in zip(datasets, datasets_names):
        task = ds.split("/")[0]
        ds = '/'.join(ds.split("/")[1:])
        path = f"{base_path}/{task}/{model}/{ds}/"
        final_path = None
        if model in paths.keys() and ds_name in paths[model].keys():
            final_path = paths[model][ds_name]
        else:
            continue
            for address, dirs, files in os.walk(path):
                for name in files:
                    full_path = os.path.join(address, name)
                    if full_path.endswith(filename):
                        final_path = copy.deepcopy(full_path)
        if final_path is not None:
            print(final_path)
            all_paths.append(final_path)
            if task == "ats":
                metric_names=['AlignScore']
            elif (ds_name in ["MMLU", "GSM8k"]):
                metric_names=["Accuracy"]
            elif task == "qa":
                metric_names=['AlignScore']
            elif task == "nmt":
                metric_names=['Comet']
            aggregated = True if ds_name in ["TruthfulQA", "TriviaQA"] else False
            df = get_table_short(final_path, dataset=ds_name, metric_names=metric_names, update_alignscore=False, aggregated=aggregated, return_score=False, upd=True, roc_auc=False, update_alignscore_sum=True, add_nll=True, mc_num=16)
            ind = np.argwhere(["P_i" in x or "MaximumSequenceProbability" in x for x in df[df.columns[-1:]].values.flatten()]).flatten()
            dfs[model][ds_name] = df.reset_index(drop=True)

./workdir/output_dllm_dec25/ats/GSAI-ML/LLaDA-1.5/['LM-Polygraph/samsum', 'simple_instruct']/2025-12-18/00-27-22/ue_manager_seed1


100%|██████████| 819/819 [02:36<00:00,  5.22it/s]


AlignScore:  0.8994821622430135
./workdir/output_dllm_dec25/nmt/GSAI-ML/LLaDA-1.5/['LM-Polygraph/wmt14', 'fren_simple_instruct']/2025-12-16/16-35-15/ue_manager_seed1


100%|██████████| 2000/2000 [02:19<00:00, 14.34it/s]


Comet:  0.8592676448002458
./workdir/output_dllm_dec25/qa/GSAI-ML/LLaDA-1.5/['LM-Polygraph/triviaqa', 'simple_instruct']/2025-12-17/16-31-56/ue_manager_seed1


100%|██████████| 2000/2000 [05:07<00:00,  6.50it/s]


AlignScore:  0.45985850962656694


In [ ]:
dfs_short = []
methods = None
model = models[0]

for key in dfs[model].keys():
    df_ = dfs[model][key]   
    if methods is None:
        methods = df_[df_.columns[-1]]
    df_ = df_.set_index(df_.columns[-1], drop=False).loc[methods]
    
    if len(dfs_short):
        dfs_short.append(df_[df_.columns[[0]]])
    else:
        dfs_short.append(df_[df_.columns[[1, 0]]])
dfs_short = pd.concat(dfs_short, axis=1).reset_index(drop=True)
dfs_short.loc[:, ('', "Mean")] = dfs_short[dfs_short.columns[1:]].mean(axis=1)
dfs_short.loc[:, ('', "Mean Rank")] = dfs_short[dfs_short.columns[1:-1]].apply(lambda x: rankdata(-x)).mean(axis=1).round(2)

In [ ]:
dfs_short.sort_values(dfs_short.columns[-1]).style.background_gradient(axis=0)

In [11]:
dfs_short.style.background_gradient(axis=0)